In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

import ngboost
from ngboost import NGBRegressor
from ngboost.distns import Normal, LogNormal, Exponential
from ngboost.scores import MLE, CRPS, LogScore

import xgboost as xgb
import lightgbm as lgb

try:
    from xgboostlss.model import XGBoostLSS
    from xgboostlss.distributions import Gaussian as XGB_Gaussian, Weibull as XGB_Weibull
    XGBOOSTLSS_AVAILABLE = True
except Exception as e:
    XGBOOSTLSS_AVAILABLE = False
    print(f" XGBoostLSS aktarılamadı ({type(e).__name__}). XGBoost yerleşik Tweedie kullanılacak.")

try:
    from lightgbmlss.model import LightGBMLSS
    from lightgbmlss.distributions import Gaussian as LGB_Gaussian, Weibull as LGB_Weibull
    LIGHTGBMLSS_AVAILABLE = True
except Exception as e:
    LIGHTGBMLSS_AVAILABLE = False
    print(f" LightGBMLSS aktarılamadı ({type(e).__name__}). LightGBM yerleşik Tweedie kullanılacak.")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f" Defter 6 Olasılıksal Ortamı (SEED={SEED}) ve PyTorch ({torch.__version__}) aktarıldı!")

 Defter 6 Olasılıksal Ortamı (SEED=42) ve PyTorch (2.5.1+cpu) aktarıldı!


In [2]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)
df.describe()

df.head(10)

Veri Boyutu: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [3]:
y_raw = df['area'].values
X_raw = df.drop(columns=['area']).copy()

cat_cols = ['month', 'day']
encoder = OrdinalEncoder()
X_encoded = X_raw.copy()
X_encoded[cat_cols] = encoder.fit_transform(X_raw[cat_cols])
X_matrix = X_encoded.values.astype(np.float64)

X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(
    X_matrix, 
    y_raw, 
    test_size=0.20, 
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("[INFO] Train/Test ayrımı ve ön işleme tamamlandı.")
print(f"X_train_encoded (Ağaç): {X_train_encoded.shape} | X_test_encoded: {X_test_encoded.shape}")
print(f"X_train_scaled (Sinir Ağı): {X_train_scaled.shape} | X_test_scaled: {X_test_scaled.shape}")
print(f"y_train (Ham): {y_train.shape}, Min={y_train.min():.2f}, Max={y_train.max():.2f}")
print(f"y_train_log (Log): {y_train_log.shape}, Min={y_train_log.min():.2f}, Max={y_train_log.max():.2f}")

[INFO] Train/Test ayrımı ve ön işleme tamamlandı.
X_train_encoded (Ağaç): (413, 12) | X_test_encoded: (104, 12)
X_train_scaled (Sinir Ağı): (413, 12) | X_test_scaled: (104, 12)
y_train (Ham): (413,), Min=0.00, Max=746.28
y_train_log (Log): (413,), Min=0.00, Max=6.62


## NGBoost Deneyleri:

### Deney 1:

In [11]:
ngb_raw = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    Base=DecisionTreeRegressor(max_depth=3),
    n_estimators=500,
    learning_rate=0.01,
    random_state=42,
    verbose=False
)

ngb_raw.fit(X_train_encoded, y_train)

dist_raw = ngb_raw.pred_dist(X_test_encoded)
y_pred_med_raw = dist_raw.loc

mc_samples_raw = dist_raw.sample(1000)
y_pred_low_raw = np.percentile(mc_samples_raw, 5, axis=0)
y_pred_up_raw = np.percentile(mc_samples_raw, 95, axis=0)

mad_raw = np.median(np.abs(y_test - y_pred_med_raw))
mae_raw = mean_absolute_error(y_test, y_pred_med_raw)
rmse_raw = np.sqrt(mean_squared_error(y_test, y_pred_med_raw))
coverage_raw = ((y_test >= y_pred_low_raw) & (y_test <= y_pred_up_raw)).mean() * 100
width_raw = np.median(y_pred_up_raw - y_pred_low_raw)

ngboost_results = []
ngboost_results.append({
    "Deney": "1 - Normal Veri (Varsayılan)",
    "MAD (ha)": mad_raw,
    "MAE (ha)": mae_raw,
    "RMSE (ha)": rmse_raw,
    "%90 Kapsama Oranı (%)": coverage_raw,
    "Medyan Aralık Genişliği (ha)": width_raw
})

print("[NGBoost - Normal Veri Varsayılan]")
print(f"MAD: {mad_raw:.4f} ha | MAE: {mae_raw:.4f} ha | RMSE: {rmse_raw:.4f} ha")
print(f"%90 Kapsama Oranı: %{coverage_raw:.2f} | Medyan Aralık Genişliği: {width_raw:.4f} ha")

[NGBoost - Normal Veri Varsayılan]
MAD: 7.6427 ha | MAE: 28.1918 ha | RMSE: 108.8518 ha
%90 Kapsama Oranı: %75.96 | Medyan Aralık Genişliği: 28.2023 ha


### Deney 2:

In [12]:
def objective_ngb_raw(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.06, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 18)
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_encoded):
        X_tr, y_tr = X_train_encoded[tr_idx], y_train[tr_idx]
        X_val, y_val = X_train_encoded[val_idx], y_train[val_idx]
        
        base_learner = DecisionTreeRegressor(
            max_depth=params["max_depth"], 
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )
        
        model = NGBRegressor(
            Dist=Normal,
            Score=MLE,
            Base=base_learner,
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            random_state=42,
            verbose=False
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        cv_mads.append(np.median(np.abs(y_val - preds)))
        
    return np.mean(cv_mads)

study_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_raw.optimize(objective_ngb_raw, n_trials=30, show_progress_bar=False)

best_p_raw = study_raw.best_params

best_base_raw = DecisionTreeRegressor(
    max_depth=best_p_raw["max_depth"],
    min_samples_leaf=best_p_raw["min_samples_leaf"],
    random_state=42
)

ngb_opt_raw = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    Base=best_base_raw,
    n_estimators=best_p_raw["n_estimators"],
    learning_rate=best_p_raw["learning_rate"],
    random_state=42,
    verbose=False
)
ngb_opt_raw.fit(X_train_encoded, y_train)

dist_opt_raw = ngb_opt_raw.pred_dist(X_test_encoded)
y_pred_med_opt = dist_opt_raw.loc

mc_samples_opt = dist_opt_raw.sample(1000)
y_pred_low_opt = np.percentile(mc_samples_opt, 5, axis=0)
y_pred_up_opt = np.percentile(mc_samples_opt, 95, axis=0)

mad_opt_raw = np.median(np.abs(y_test - y_pred_med_opt))
mae_opt_raw = mean_absolute_error(y_test, y_pred_med_opt)
rmse_opt_raw = np.sqrt(mean_squared_error(y_test, y_pred_med_opt))
coverage_opt_raw = ((y_test >= y_pred_low_opt) & (y_test <= y_pred_up_opt)).mean() * 100
width_opt_raw = np.median(y_pred_up_opt - y_pred_low_opt)

ngboost_results.append({
    "Deney Aşaması": "2 - Normal Veri (Bayesyen Optuna HPO)",
    "Dağılım Ailesi": "Normal (Gaussian)",
    "MAD (ha)": mad_opt_raw,
    "MAE (ha)": mae_opt_raw,
    "RMSE (ha)": rmse_opt_raw,
    "%90 Kapsama Oranı (%)": coverage_opt_raw,
    "Medyan Aralık Genişliği (ha)": width_opt_raw
})

print(f"-> Altın Hiperparametreler  : {best_p_raw}")
print(f"-> Nokta Tahmin Performansı : MAD = {mad_opt_raw:.4f} ha | MAE = {mae_opt_raw:.4f} ha | RMSE = {rmse_opt_raw:.4f} ha")
print(f"-> Belirsizlik Kalibrasyonu : %90 Kapsama = %{coverage_opt_raw:.2f} | Medyan Bant Genişliği = {width_opt_raw:.4f} ha")

df_stage2 = pd.DataFrame(ngboost_results)
styled_stage2 = (
    df_stage2.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Aralık Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Dağılım Ailesi"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage2)

-> Altın Hiperparametreler  : {'n_estimators': 630, 'learning_rate': 0.005983375935652698, 'max_depth': 3, 'min_samples_leaf': 3}
-> Nokta Tahmin Performansı : MAD = 7.7541 ha | MAE = 27.6452 ha | RMSE = 110.0994 ha
-> Belirsizlik Kalibrasyonu : %90 Kapsama = %84.62 | Medyan Bant Genişliği = 48.6028 ha


,Deney,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Aralık Genişliği (ha),Deney Aşaması,Dağılım Ailesi
0,1 - Normal Veri (Varsayılan),7.6427,28.1918,108.8518,75.96%,28.2023,nan,nan
1,nan,7.7541,27.6452,110.0994,84.62%,48.6028,2 - Normal Veri (Bayesyen Optuna HPO),Normal (Gaussian)


### Deney 3:

In [13]:
for item in ngboost_results:
    if "Deney" in item:
        item["Deney Aşaması"] = item.pop("Deney")
    if "Dağılım Ailesi" not in item or pd.isna(item.get("Dağılım Ailesi")):
        item["Dağılım Ailesi"] = "Normal (Gaussian)"

ngb_log_def = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    Base=DecisionTreeRegressor(max_depth=3),
    n_estimators=500,
    learning_rate=0.01,
    random_state=42,
    verbose=False
)

ngb_log_def.fit(X_train_encoded, y_train_log)

dist_log_def = ngb_log_def.pred_dist(X_test_encoded)

mc_samples_log_def = dist_log_def.sample(1000)
mc_samples_ha_def = np.expm1(mc_samples_log_def)
mc_samples_ha_def = np.clip(mc_samples_ha_def, 0, None)

y_pred_med_log_def = np.percentile(mc_samples_ha_def, 50, axis=0)
y_pred_low_log_def = np.percentile(mc_samples_ha_def, 5, axis=0)
y_pred_up_log_def = np.percentile(mc_samples_ha_def, 95, axis=0)

mad_log_def = np.median(np.abs(y_test - y_pred_med_log_def))
mae_log_def = mean_absolute_error(y_test, y_pred_med_log_def)
rmse_log_def = np.sqrt(mean_squared_error(y_test, y_pred_med_log_def))
coverage_log_def = ((y_test >= y_pred_low_log_def) & (y_test <= y_pred_up_log_def)).mean() * 100
width_log_def = np.median(y_pred_up_log_def - y_pred_low_log_def)

ngboost_results.append({
    "Deney Aşaması": "3 - Log Veri ln(y+1) (Varsayılan)",
    "Dağılım Ailesi": "Log-Normal Sentezi",
    "MAD (ha)": mad_log_def,
    "MAE (ha)": mae_log_def,
    "RMSE (ha)": rmse_log_def,
    "%90 Kapsama Oranı (%)": coverage_log_def,
    "Medyan Aralık Genişliği (ha)": width_log_def
})

print(f"-> Nokta Tahmin Performansı : MAD = {mad_log_def:.4f} ha | MAE = {mae_log_def:.4f} ha | RMSE = {rmse_log_def:.4f} ha")
print(f"-> Belirsizlik Kalibrasyonu : %90 Kapsama = %{coverage_log_def:.2f} | Medyan Bant Genişliği = {width_log_def:.4f} ha")

df_stage3 = pd.DataFrame(ngboost_results)
styled_stage3 = (
    df_stage3[['Deney Aşaması', 'Dağılım Ailesi', 'MAD (ha)', 'MAE (ha)', 'RMSE (ha)', '%90 Kapsama Oranı (%)', 'Medyan Aralık Genişliği (ha)']]
    .style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Aralık Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Dağılım Ailesi"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage3)

-> Nokta Tahmin Performansı : MAD = 2.4323 ha | MAE = 19.9752 ha | RMSE = 109.9121 ha
-> Belirsizlik Kalibrasyonu : %90 Kapsama = %77.88 | Medyan Bant Genişliği = 13.0915 ha


,Deney Aşaması,Dağılım Ailesi,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Aralık Genişliği (ha)
0,1 - Normal Veri (Varsayılan),Normal (Gaussian),7.6427,28.1918,108.8518,75.96%,28.2023
1,2 - Normal Veri (Bayesyen Optuna HPO),Normal (Gaussian),7.7541,27.6452,110.0994,84.62%,48.6028
2,3 - Log Veri ln(y+1) (Varsayılan),Log-Normal Sentezi,2.4323,19.9752,109.9121,77.88%,13.0915


In [14]:
def objective_ngb_log(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_encoded):
        X_tr, y_tr_log = X_train_encoded[tr_idx], y_train_log[tr_idx]
        X_val, y_val_raw = X_train_encoded[val_idx], y_train[val_idx]
        
        base_learner = DecisionTreeRegressor(
            max_depth=params["max_depth"], 
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )
        
        model = NGBRegressor(
            Dist=Normal,
            Score=MLE,
            Base=base_learner,
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            random_state=42,
            verbose=False
        )
        model.fit(X_tr, y_tr_log)
        
        dist_val = model.pred_dist(X_val)
        mc_val = np.expm1(dist_val.sample(500))
        preds_ha = np.percentile(np.clip(mc_val, 0, None), 50, axis=0)
        cv_mads.append(np.median(np.abs(y_val_raw - preds_ha)))
        
    return np.mean(cv_mads)

study_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_log.optimize(objective_ngb_log, n_trials=25, show_progress_bar=False)

best_p_log = study_log.best_params

best_base_log = DecisionTreeRegressor(
    max_depth=best_p_log["max_depth"],
    min_samples_leaf=best_p_log["min_samples_leaf"],
    random_state=42
)

ngb_opt_log = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    Base=best_base_log,
    n_estimators=best_p_log["n_estimators"],
    learning_rate=best_p_log["learning_rate"],
    random_state=42,
    verbose=False
)
ngb_opt_log.fit(X_train_encoded, y_train_log)

dist_opt_log = ngb_opt_log.pred_dist(X_test_encoded)
mc_samples_opt_log = np.expm1(dist_opt_log.sample(1000))
mc_samples_opt_log = np.clip(mc_samples_opt_log, 0, None)

y_pred_med_opt_log = np.percentile(mc_samples_opt_log, 50, axis=0)
y_pred_low_opt_log = np.percentile(mc_samples_opt_log, 5, axis=0)
y_pred_up_opt_log = np.percentile(mc_samples_opt_log, 95, axis=0)

mad_opt_log = np.median(np.abs(y_test - y_pred_med_opt_log))
mae_opt_log = mean_absolute_error(y_test, y_pred_med_opt_log)
rmse_opt_log = np.sqrt(mean_squared_error(y_test, y_pred_med_opt_log))
coverage_opt_log = ((y_test >= y_pred_low_opt_log) & (y_test <= y_pred_up_opt_log)).mean() * 100
width_opt_log = np.median(y_pred_up_opt_log - y_pred_low_opt_log)

ngboost_results.append({
    "Deney Aşaması": "4 - Log Veri ln(y+1) (Bayesyen Optuna HPO)",
    "Dağılım Ailesi": "Log-Normal Sentezi",
    "MAD (ha)": mad_opt_log,
    "MAE (ha)": mae_opt_log,
    "RMSE (ha)": rmse_opt_log,
    "%90 Kapsama Oranı (%)": coverage_opt_log,
    "Medyan Aralık Genişliği (ha)": width_opt_log
})

print(f"-> Altın Hiperparametreler  : {best_p_log}")
print(f"-> Nokta Tahmin Performansı : MAD = {mad_opt_log:.4f} ha | MAE = {mae_opt_log:.4f} ha | RMSE = {rmse_opt_log:.4f} ha")
print(f"-> Belirsizlik Kalibrasyonu : %90 Kapsama = %{coverage_opt_log:.2f} | Medyan Bant Genişliği = {width_opt_log:.4f} ha")

df_ngboost_final = pd.DataFrame(ngboost_results)
styled_ngboost_final = (
    df_ngboost_final[['Deney Aşaması', 'Dağılım Ailesi', 'MAD (ha)', 'MAE (ha)', 'RMSE (ha)', '%90 Kapsama Oranı (%)', 'Medyan Aralık Genişliği (ha)']]
    .style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Aralık Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Dağılım Ailesi"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_ngboost_final)

-> Altın Hiperparametreler  : {'n_estimators': 228, 'learning_rate': 0.020959316039143597, 'max_depth': 2, 'min_samples_leaf': 5}
-> Nokta Tahmin Performansı : MAD = 2.4511 ha | MAE = 19.7249 ha | RMSE = 109.8880 ha
-> Belirsizlik Kalibrasyonu : %90 Kapsama = %84.62 | Medyan Bant Genişliği = 21.1563 ha


,Deney Aşaması,Dağılım Ailesi,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Aralık Genişliği (ha)
0,1 - Normal Veri (Varsayılan),Normal (Gaussian),7.6427,28.1918,108.8518,75.96%,28.2023
1,2 - Normal Veri (Bayesyen Optuna HPO),Normal (Gaussian),7.7541,27.6452,110.0994,84.62%,48.6028
2,3 - Log Veri ln(y+1) (Varsayılan),Log-Normal Sentezi,2.4323,19.9752,109.9121,77.88%,13.0915
3,4 - Log Veri ln(y+1) (Bayesyen Optuna HPO),Log-Normal Sentezi,2.4511,19.7249,109.8880,84.62%,21.1563


## XGBoostLSS / LightGBMLSS Deneyleri:

### Deney 1:

In [17]:
dtrain_xgb = xgb.DMatrix(X_train_encoded, label=y_train)
dtest_xgb = xgb.DMatrix(X_test_encoded)

xgblss_raw = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
params_xgb_def = {"eta": 0.05, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8}
xgblss_raw.train(params_xgb_def, dtrain_xgb, num_boost_round=150)

dist_params_xgb = xgblss_raw.predict(dtest_xgb, pred_type="parameters")
mu_xgb_raw = dist_params_xgb["loc"].values if hasattr(dist_params_xgb, "loc") else dist_params_xgb.iloc[:, 0].values
sigma_xgb_raw = dist_params_xgb["scale"].values if hasattr(dist_params_xgb, "scale") else dist_params_xgb.iloc[:, 1].values

mc_xgb_raw = stats.norm.rvs(loc=mu_xgb_raw, scale=sigma_xgb_raw, size=(1000, len(mu_xgb_raw)), random_state=42)
y_pred_med_xgb = np.percentile(mc_xgb_raw, 50, axis=0)
y_pred_low_xgb = np.percentile(mc_xgb_raw, 5, axis=0)
y_pred_up_xgb = np.percentile(mc_xgb_raw, 95, axis=0)

mad_xgb = np.median(np.abs(y_test - y_pred_med_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_med_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_med_xgb))
cov_xgb = ((y_test >= y_pred_low_xgb) & (y_test <= y_pred_up_xgb)).mean() * 100
width_xgb = np.median(y_pred_up_xgb - y_pred_low_xgb)


dtrain_lgb = lgb.Dataset(X_train_encoded, label=y_train)

lgblss_raw = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
params_lgb_def = {"learning_rate": 0.05, "max_depth": 3, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1}
lgblss_raw.train(params_lgb_def, dtrain_lgb, num_boost_round=150)

dist_params_lgb = lgblss_raw.predict(X_test_encoded, pred_type="parameters")
mu_lgb_raw = dist_params_lgb["loc"].values if hasattr(dist_params_lgb, "loc") else dist_params_lgb.iloc[:, 0].values
sigma_lgb_raw = dist_params_lgb["scale"].values if hasattr(dist_params_lgb, "scale") else dist_params_lgb.iloc[:, 1].values

mc_lgb_raw = stats.norm.rvs(loc=mu_lgb_raw, scale=sigma_lgb_raw, size=(1000, len(mu_lgb_raw)), random_state=42)
y_pred_med_lgb = np.percentile(mc_lgb_raw, 50, axis=0)
y_pred_low_lgb = np.percentile(mc_lgb_raw, 5, axis=0)
y_pred_up_lgb = np.percentile(mc_lgb_raw, 95, axis=0)

mad_lgb = np.median(np.abs(y_test - y_pred_med_lgb))
mae_lgb = mean_absolute_error(y_test, y_pred_med_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_med_lgb))
cov_lgb = ((y_test >= y_pred_low_lgb) & (y_test <= y_pred_up_lgb)).mean() * 100
width_lgb = np.median(y_pred_up_lgb - y_pred_low_lgb)

if 'gamlss_results' not in globals():
    gamlss_results = []

gamlss_results.append({
    "Deney Aşaması": "1 - Normal Veri (Varsayılan)",
    "Model Motoru": "XGBoostLSS (Newton GAMLSS)",
    "MAD (ha)": mad_xgb,
    "MAE (ha)": mae_xgb,
    "RMSE (ha)": rmse_xgb,
    "%90 Kapsama Oranı (%)": cov_xgb,
    "Medyan Bant Genişliği (ha)": width_xgb
})

gamlss_results.append({
    "Deney Aşaması": "1 - Normal Veri (Varsayılan)",
    "Model Motoru": "LightGBMLSS (Newton GAMLSS)",
    "MAD (ha)": mad_lgb,
    "MAE (ha)": mae_lgb,
    "RMSE (ha)": rmse_lgb,
    "%90 Kapsama Oranı (%)": cov_lgb,
    "Medyan Bant Genişliği (ha)": width_lgb
})

print(f"XGBoostLSS  -> MAD: {mad_xgb:.4f} ha | RMSE: {rmse_xgb:.4f} ha | %90 Kapsama: %{cov_xgb:.2f} | Genişlik: {width_xgb:.4f} ha")
print(f"LightGBMLSS -> MAD: {mad_lgb:.4f} ha | RMSE: {rmse_lgb:.4f} ha | %90 Kapsama: %{cov_lgb:.2f} | Genişlik: {width_lgb:.4f} ha")

df_stage1_gamlss = pd.DataFrame(gamlss_results)
styled_stage1_gamlss = (
    df_stage1_gamlss.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage1_gamlss)

XGBoostLSS  -> MAD: 3.2458 ha | RMSE: 109.6175 ha | %90 Kapsama: %80.77 | Genişlik: 19.7509 ha
LightGBMLSS -> MAD: 6.6514 ha | RMSE: 108.8111 ha | %90 Kapsama: %84.62 | Genişlik: 33.7721 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),XGBoostLSS (Newton GAMLSS),3.2458,20.0613,109.6175,80.77%,19.7509
1,1 - Normal Veri (Varsayılan),LightGBMLSS (Newton GAMLSS),6.6514,24.1396,108.8111,84.62%,33.7721


### Deney 2:

In [18]:
def objective_xgb_gamlss_raw(trial):
    params = {
        "eta": trial.suggest_float("eta", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95)
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    for tr_idx, val_idx in kf.split(X_train_encoded):
        dtr = xgb.DMatrix(X_train_encoded[tr_idx], label=y_train[tr_idx])
        dval = xgb.DMatrix(X_train_encoded[val_idx])
        model = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
        model.train(params, dtr, num_boost_round=120)
        dist_p = model.predict(dval, pred_type="parameters")
        mu_val = dist_p["loc"].values if hasattr(dist_p, "loc") else dist_p.iloc[:, 0].values
        cv_mads.append(np.median(np.abs(y_train[val_idx] - mu_val)))
    return np.mean(cv_mads)

study_xgb_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb_raw.optimize(objective_xgb_gamlss_raw, n_trials=20, show_progress_bar=False)
best_xgb_p_raw = study_xgb_raw.best_params

xgblss_opt_raw = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
xgblss_opt_raw.train(best_xgb_p_raw, dtrain_xgb, num_boost_round=120)

dist_p_xgb_opt = xgblss_opt_raw.predict(dtest_xgb, pred_type="parameters")
mu_xgb_opt = dist_p_xgb_opt["loc"].values if hasattr(dist_p_xgb_opt, "loc") else dist_p_xgb_opt.iloc[:, 0].values
sigma_xgb_opt = dist_p_xgb_opt["scale"].values if hasattr(dist_p_xgb_opt, "scale") else dist_p_xgb_opt.iloc[:, 1].values

mc_xgb_opt = stats.norm.rvs(loc=mu_xgb_opt, scale=sigma_xgb_opt, size=(1000, len(mu_xgb_opt)), random_state=42)
y_med_xgb_opt = np.percentile(mc_xgb_opt, 50, axis=0)
y_low_xgb_opt = np.percentile(mc_xgb_opt, 5, axis=0)
y_up_xgb_opt = np.percentile(mc_xgb_opt, 95, axis=0)

mad_xgb_opt = np.median(np.abs(y_test - y_med_xgb_opt))
mae_xgb_opt = mean_absolute_error(y_test, y_med_xgb_opt)
rmse_xgb_opt = np.sqrt(mean_squared_error(y_test, y_med_xgb_opt))
cov_xgb_opt = ((y_test >= y_low_xgb_opt) & (y_test <= y_up_xgb_opt)).mean() * 100
width_xgb_opt = np.median(y_up_xgb_opt - y_low_xgb_opt)


def objective_lgb_gamlss_raw(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 0.95),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 0.95),
        "bagging_freq": 1
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    for tr_idx, val_idx in kf.split(X_train_encoded):
        dtr = lgb.Dataset(X_train_encoded[tr_idx], label=y_train[tr_idx])
        model = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
        model.train(params, dtr, num_boost_round=120)
        dist_p = model.predict(X_train_encoded[val_idx], pred_type="parameters")
        mu_val = dist_p["loc"].values if hasattr(dist_p, "loc") else dist_p.iloc[:, 0].values
        cv_mads.append(np.median(np.abs(y_train[val_idx] - mu_val)))
    return np.mean(cv_mads)

study_lgb_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgb_raw.optimize(objective_lgb_gamlss_raw, n_trials=20, show_progress_bar=False)
best_lgb_p_raw = study_lgb_raw.best_params
best_lgb_p_raw["bagging_freq"] = 1

lgblss_opt_raw = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
lgblss_opt_raw.train(best_lgb_p_raw, dtrain_lgb, num_boost_round=120)

dist_p_lgb_opt = lgblss_opt_raw.predict(X_test_encoded, pred_type="parameters")
mu_lgb_opt = dist_p_lgb_opt["loc"].values if hasattr(dist_p_lgb_opt, "loc") else dist_p_lgb_opt.iloc[:, 0].values
sigma_lgb_opt = dist_p_lgb_opt["scale"].values if hasattr(dist_p_lgb_opt, "scale") else dist_p_lgb_opt.iloc[:, 1].values

mc_lgb_opt = stats.norm.rvs(loc=mu_lgb_opt, scale=sigma_lgb_opt, size=(1000, len(mu_lgb_opt)), random_state=42)
y_med_lgb_opt = np.percentile(mc_lgb_opt, 50, axis=0)
y_low_lgb_opt = np.percentile(mc_lgb_opt, 5, axis=0)
y_up_lgb_opt = np.percentile(mc_lgb_opt, 95, axis=0)

mad_lgb_opt = np.median(np.abs(y_test - y_med_lgb_opt))
mae_lgb_opt = mean_absolute_error(y_test, y_med_lgb_opt)
rmse_lgb_opt = np.sqrt(mean_squared_error(y_test, y_med_lgb_opt))
cov_lgb_opt = ((y_test >= y_low_lgb_opt) & (y_test <= y_up_lgb_opt)).mean() * 100
width_lgb_opt = np.median(y_up_lgb_opt - y_low_lgb_opt)

gamlss_results.append({
    "Deney Aşaması": "2 - Normal Veri (Bayesyen Optuna HPO)",
    "Model Motoru": "XGBoostLSS (Newton GAMLSS)",
    "MAD (ha)": mad_xgb_opt,
    "MAE (ha)": mae_xgb_opt,
    "RMSE (ha)": rmse_xgb_opt,
    "%90 Kapsama Oranı (%)": cov_xgb_opt,
    "Medyan Bant Genişliği (ha)": width_xgb_opt
})

gamlss_results.append({
    "Deney Aşaması": "2 - Normal Veri (Bayesyen Optuna HPO)",
    "Model Motoru": "LightGBMLSS (Newton GAMLSS)",
    "MAD (ha)": mad_lgb_opt,
    "MAE (ha)": mae_lgb_opt,
    "RMSE (ha)": rmse_lgb_opt,
    "%90 Kapsama Oranı (%)": cov_lgb_opt,
    "Medyan Bant Genişliği (ha)": width_lgb_opt
})

print(f"XGBoostLSS  -> MAD: {mad_xgb_opt:.4f} ha | MAE: {mae_xgb_opt:.4f} ha | RMSE: {rmse_xgb_opt:.4f} ha | %90 Kapsama: %{cov_xgb_opt:.2f} | Genişlik: {width_xgb_opt:.4f} ha")
print(f"LightGBMLSS -> MAD: {mad_lgb_opt:.4f} ha | MAE: {mae_lgb_opt:.4f} ha | RMSE: {rmse_lgb_opt:.4f} ha | %90 Kapsama: %{cov_lgb_opt:.2f} | Genişlik: {width_lgb_opt:.4f} ha")

df_stage2_gamlss = pd.DataFrame(gamlss_results)
styled_stage2_gamlss = (
    df_stage2_gamlss.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage2_gamlss)

XGBoostLSS  -> MAD: 2.6102 ha | MAE: 19.8047 ha | RMSE: 109.6993 ha | %90 Kapsama: %76.92 | Genişlik: 13.5516 ha
LightGBMLSS -> MAD: 6.6598 ha | MAE: 23.2392 ha | RMSE: 108.5058 ha | %90 Kapsama: %41.35 | Genişlik: 9.7350 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),XGBoostLSS (Newton GAMLSS),3.2458,20.0613,109.6175,80.77%,19.7509
1,1 - Normal Veri (Varsayılan),LightGBMLSS (Newton GAMLSS),6.6514,24.1396,108.8111,84.62%,33.7721
2,2 - Normal Veri (Bayesyen Optuna HPO),XGBoostLSS (Newton GAMLSS),2.6102,19.8047,109.6993,76.92%,13.5516
3,2 - Normal Veri (Bayesyen Optuna HPO),LightGBMLSS (Newton GAMLSS),6.6598,23.2392,108.5058,41.35%,9.7350


### Deney 3:

In [19]:
dtrain_xgb_log = xgb.DMatrix(X_train_encoded, label=y_train_log)
xgblss_log_def = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
params_xgb_def_log = {"eta": 0.05, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8}
xgblss_log_def.train(params_xgb_def_log, dtrain_xgb_log, num_boost_round=150)

dist_p_xgb_log = xgblss_log_def.predict(dtest_xgb, pred_type="parameters")
mu_xgb_log = dist_p_xgb_log["loc"].values if hasattr(dist_p_xgb_log, "loc") else dist_p_xgb_log.iloc[:, 0].values
sigma_xgb_log = dist_p_xgb_log["scale"].values if hasattr(dist_p_xgb_log, "scale") else dist_p_xgb_log.iloc[:, 1].values

mc_xgb_log = stats.norm.rvs(loc=mu_xgb_log, scale=sigma_xgb_log, size=(1000, len(mu_xgb_log)), random_state=42)
mc_xgb_ha_def = np.clip(np.expm1(mc_xgb_log), 0, None)

y_med_xgb_log_def = np.percentile(mc_xgb_ha_def, 50, axis=0)
y_low_xgb_log_def = np.percentile(mc_xgb_ha_def, 5, axis=0)
y_up_xgb_log_def = np.percentile(mc_xgb_ha_def, 95, axis=0)

mad_xgb_log_def = np.median(np.abs(y_test - y_med_xgb_log_def))
mae_xgb_log_def = mean_absolute_error(y_test, y_med_xgb_log_def)
rmse_xgb_log_def = np.sqrt(mean_squared_error(y_test, y_med_xgb_log_def))
cov_xgb_log_def = ((y_test >= y_low_xgb_log_def) & (y_test <= y_up_xgb_log_def)).mean() * 100
width_xgb_log_def = np.median(y_up_xgb_log_def - y_low_xgb_log_def)


dtrain_lgb_log = lgb.Dataset(X_train_encoded, label=y_train_log)
lgblss_log_def = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
params_lgb_def_log = {"learning_rate": 0.05, "max_depth": 3, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1}
lgblss_log_def.train(params_lgb_def_log, dtrain_lgb_log, num_boost_round=150)

dist_p_lgb_log = lgblss_log_def.predict(X_test_encoded, pred_type="parameters")
mu_lgb_log = dist_p_lgb_log["loc"].values if hasattr(dist_p_lgb_log, "loc") else dist_p_lgb_log.iloc[:, 0].values
sigma_lgb_log = dist_p_lgb_log["scale"].values if hasattr(dist_p_lgb_log, "scale") else dist_p_lgb_log.iloc[:, 1].values

mc_lgb_log = stats.norm.rvs(loc=mu_lgb_log, scale=sigma_lgb_log, size=(1000, len(mu_lgb_log)), random_state=42)
mc_lgb_ha_def = np.clip(np.expm1(mc_lgb_log), 0, None)

y_med_lgb_log_def = np.percentile(mc_lgb_ha_def, 50, axis=0)
y_low_lgb_log_def = np.percentile(mc_lgb_ha_def, 5, axis=0)
y_up_lgb_log_def = np.percentile(mc_lgb_ha_def, 95, axis=0)

mad_lgb_log_def = np.median(np.abs(y_test - y_med_lgb_log_def))
mae_lgb_log_def = mean_absolute_error(y_test, y_med_lgb_log_def)
rmse_lgb_log_def = np.sqrt(mean_squared_error(y_test, y_med_lgb_log_def))
cov_lgb_log_def = ((y_test >= y_low_lgb_log_def) & (y_test <= y_up_lgb_log_def)).mean() * 100
width_lgb_log_def = np.median(y_up_lgb_log_def - y_low_lgb_log_def)

gamlss_results.append({
    "Deney Aşaması": "3 - Log Veri ln(y+1) (Varsayılan)",
    "Model Motoru": "XGBoostLSS (Newton GAMLSS)",
    "MAD (ha)": mad_xgb_log_def,
    "MAE (ha)": mae_xgb_log_def,
    "RMSE (ha)": rmse_xgb_log_def,
    "%90 Kapsama Oranı (%)": cov_xgb_log_def,
    "Medyan Bant Genişliği (ha)": width_xgb_log_def
})

gamlss_results.append({
    "Deney Aşaması": "3 - Log Veri ln(y+1) (Varsayılan)",
    "Model Motoru": "LightGBMLSS (Newton GAMLSS)",
    "MAD (ha)": mad_lgb_log_def,
    "MAE (ha)": mae_lgb_log_def,
    "RMSE (ha)": rmse_lgb_log_def,
    "%90 Kapsama Oranı (%)": cov_lgb_log_def,
    "Medyan Bant Genişliği (ha)": width_lgb_log_def
})

print(f"XGBoostLSS  -> MAD: {mad_xgb_log_def:.4f} ha | MAE: {mae_xgb_log_def:.4f} ha | RMSE: {rmse_xgb_log_def:.4f} ha | %90 Kapsama: %{cov_xgb_log_def:.2f} | Genişlik: {width_xgb_log_def:.4f} ha")
print(f"LightGBMLSS -> MAD: {mad_lgb_log_def:.4f} ha | MAE: {mae_lgb_log_def:.4f} ha | RMSE: {rmse_lgb_log_def:.4f} ha | %90 Kapsama: %{cov_lgb_log_def:.2f} | Genişlik: {width_lgb_log_def:.4f} ha")

df_stage3_gamlss = pd.DataFrame(gamlss_results)
styled_stage3_gamlss = (
    df_stage3_gamlss.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage3_gamlss)

XGBoostLSS  -> MAD: 1.9897 ha | MAE: 19.7140 ha | RMSE: 109.8943 ha | %90 Kapsama: %72.12 | Genişlik: 9.8532 ha
LightGBMLSS -> MAD: 2.1941 ha | MAE: 19.8585 ha | RMSE: 109.8721 ha | %90 Kapsama: %73.08 | Genişlik: 10.0188 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),XGBoostLSS (Newton GAMLSS),3.2458,20.0613,109.6175,80.77%,19.7509
1,1 - Normal Veri (Varsayılan),LightGBMLSS (Newton GAMLSS),6.6514,24.1396,108.8111,84.62%,33.7721
2,2 - Normal Veri (Bayesyen Optuna HPO),XGBoostLSS (Newton GAMLSS),2.6102,19.8047,109.6993,76.92%,13.5516
3,2 - Normal Veri (Bayesyen Optuna HPO),LightGBMLSS (Newton GAMLSS),6.6598,23.2392,108.5058,41.35%,9.7350
4,3 - Log Veri ln(y+1) (Varsayılan),XGBoostLSS (Newton GAMLSS),1.9897,19.7140,109.8943,72.12%,9.8532
5,3 - Log Veri ln(y+1) (Varsayılan),LightGBMLSS (Newton GAMLSS),2.1941,19.8585,109.8721,73.08%,10.0188


### Deney 4:

In [20]:
def objective_xgb_gamlss_log(trial):
    params = {
        "eta": trial.suggest_float("eta", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95)
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    for tr_idx, val_idx in kf.split(X_train_encoded):
        dtr = xgb.DMatrix(X_train_encoded[tr_idx], label=y_train_log[tr_idx])
        dval = xgb.DMatrix(X_train_encoded[val_idx])
        model = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
        model.train(params, dtr, num_boost_round=120)
        dist_p = model.predict(dval, pred_type="parameters")
        mu_log = dist_p["loc"].values if hasattr(dist_p, "loc") else dist_p.iloc[:, 0].values
        preds_ha = np.clip(np.expm1(mu_log), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha)))
    return np.mean(cv_mads)

study_xgb_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb_log.optimize(objective_xgb_gamlss_log, n_trials=20, show_progress_bar=False)
best_xgb_p_log = study_xgb_log.best_params

xgblss_opt_log = XGBoostLSS(XGB_Gaussian.Gaussian(stabilization="None"))
xgblss_opt_log.train(best_xgb_p_log, dtrain_xgb_log, num_boost_round=120)

dist_p_xgb_opt_log = xgblss_opt_log.predict(dtest_xgb, pred_type="parameters")
mu_xgb_opt_log = dist_p_xgb_opt_log["loc"].values if hasattr(dist_p_xgb_opt_log, "loc") else dist_p_xgb_opt_log.iloc[:, 0].values
sigma_xgb_opt_log = dist_p_xgb_opt_log["scale"].values if hasattr(dist_p_xgb_opt_log, "scale") else dist_p_xgb_opt_log.iloc[:, 1].values

mc_xgb_opt_log = stats.norm.rvs(loc=mu_xgb_opt_log, scale=sigma_xgb_opt_log, size=(1000, len(mu_xgb_opt_log)), random_state=42)
mc_xgb_ha_opt = np.clip(np.expm1(mc_xgb_opt_log), 0, None)

y_med_xgb_opt_log = np.percentile(mc_xgb_ha_opt, 50, axis=0)
y_low_xgb_opt_log = np.percentile(mc_xgb_ha_opt, 5, axis=0)
y_up_xgb_opt_log = np.percentile(mc_xgb_ha_opt, 95, axis=0)

mad_xgb_opt_log = np.median(np.abs(y_test - y_med_xgb_opt_log))
mae_xgb_opt_log = mean_absolute_error(y_test, y_med_xgb_opt_log)
rmse_xgb_opt_log = np.sqrt(mean_squared_error(y_test, y_med_xgb_opt_log))
cov_xgb_opt_log = ((y_test >= y_low_xgb_opt_log) & (y_test <= y_up_xgb_opt_log)).mean() * 100
width_xgb_opt_log = np.median(y_up_xgb_opt_log - y_low_xgb_opt_log)


def objective_lgb_gamlss_log(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 0.95),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 0.95),
        "bagging_freq": 1
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    for tr_idx, val_idx in kf.split(X_train_encoded):
        dtr = lgb.Dataset(X_train_encoded[tr_idx], label=y_train_log[tr_idx])
        model = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
        model.train(params, dtr, num_boost_round=120)
        dist_p = model.predict(X_train_encoded[val_idx], pred_type="parameters")
        mu_log = dist_p["loc"].values if hasattr(dist_p, "loc") else dist_p.iloc[:, 0].values
        preds_ha = np.clip(np.expm1(mu_log), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha)))
    return np.mean(cv_mads)

study_lgb_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgb_log.optimize(objective_lgb_gamlss_log, n_trials=20, show_progress_bar=False)
best_lgb_p_log = study_lgb_log.best_params
best_lgb_p_log["bagging_freq"] = 1

lgblss_opt_log = LightGBMLSS(LGB_Gaussian.Gaussian(stabilization="None"))
lgblss_opt_log.train(best_lgb_p_log, dtrain_lgb_log, num_boost_round=120)

dist_p_lgb_opt_log = lgblss_opt_log.predict(X_test_encoded, pred_type="parameters")
mu_lgb_opt_log = dist_p_lgb_opt_log["loc"].values if hasattr(dist_p_lgb_opt_log, "loc") else dist_p_lgb_opt_log.iloc[:, 0].values
sigma_lgb_opt_log = dist_p_lgb_opt_log["scale"].values if hasattr(dist_p_lgb_opt_log, "scale") else dist_p_lgb_opt_log.iloc[:, 1].values

mc_lgb_opt_log = stats.norm.rvs(loc=mu_lgb_opt_log, scale=sigma_lgb_opt_log, size=(1000, len(mu_lgb_opt_log)), random_state=42)
mc_lgb_ha_opt = np.clip(np.expm1(mc_lgb_opt_log), 0, None)

y_med_lgb_opt_log = np.percentile(mc_lgb_ha_opt, 50, axis=0)
y_low_lgb_opt_log = np.percentile(mc_lgb_ha_opt, 5, axis=0)
y_up_lgb_opt_log = np.percentile(mc_lgb_ha_opt, 95, axis=0)

mad_lgb_opt_log = np.median(np.abs(y_test - y_med_lgb_opt_log))
mae_lgb_opt_log = mean_absolute_error(y_test, y_med_lgb_opt_log)
rmse_lgb_opt_log = np.sqrt(mean_squared_error(y_test, y_med_lgb_opt_log))
cov_lgb_opt_log = ((y_test >= y_low_lgb_opt_log) & (y_test <= y_up_lgb_opt_log)).mean() * 100
width_lgb_opt_log = np.median(y_up_lgb_opt_log - y_low_lgb_opt_log)

gamlss_results.append({
    "Deney Aşaması": "4 - Log Veri ln(y+1) (Bayesyen Optuna HPO)",
    "Model Motoru": "XGBoostLSS (Newton GAMLSS)",
    "MAD (ha)": mad_xgb_opt_log,
    "MAE (ha)": mae_xgb_opt_log,
    "RMSE (ha)": rmse_xgb_opt_log,
    "%90 Kapsama Oranı (%)": cov_xgb_opt_log,
    "Medyan Bant Genişliği (ha)": width_xgb_opt_log
})

gamlss_results.append({
    "Deney Aşaması": "4 - Log Veri ln(y+1) (Bayesyen Optuna HPO)",
    "Model Motoru": "LightGBMLSS (Newton GAMLSS)",
    "MAD (ha)": mad_lgb_opt_log,
    "MAE (ha)": mae_lgb_opt_log,
    "RMSE (ha)": rmse_lgb_opt_log,
    "%90 Kapsama Oranı (%)": cov_lgb_opt_log,
    "Medyan Bant Genişliği (ha)": width_lgb_opt_log
})

print(f"XGBoostLSS  -> MAD: {mad_xgb_opt_log:.4f} ha | MAE: {mae_xgb_opt_log:.4f} ha | RMSE: {rmse_xgb_opt_log:.4f} ha | %90 Kapsama: %{cov_xgb_opt_log:.2f} | Genişlik: {width_xgb_opt_log:.4f} ha")
print(f"LightGBMLSS -> MAD: {mad_lgb_opt_log:.4f} ha | MAE: {mae_lgb_opt_log:.4f} ha | RMSE: {rmse_lgb_opt_log:.4f} ha | %90 Kapsama: %{cov_lgb_opt_log:.2f} | Genişlik: {width_lgb_opt_log:.4f} ha")

df_gamlss_final = pd.DataFrame(gamlss_results)
styled_gamlss_final = (
    df_gamlss_final.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_gamlss_final)

XGBoostLSS  -> MAD: 1.9181 ha | MAE: 19.6878 ha | RMSE: 109.9681 ha | %90 Kapsama: %67.31 | Genişlik: 8.5235 ha
LightGBMLSS -> MAD: 1.6541 ha | MAE: 19.7435 ha | RMSE: 109.9673 ha | %90 Kapsama: %85.58 | Genişlik: 17.3440 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),XGBoostLSS (Newton GAMLSS),3.2458,20.0613,109.6175,80.77%,19.7509
1,1 - Normal Veri (Varsayılan),LightGBMLSS (Newton GAMLSS),6.6514,24.1396,108.8111,84.62%,33.7721
2,2 - Normal Veri (Bayesyen Optuna HPO),XGBoostLSS (Newton GAMLSS),2.6102,19.8047,109.6993,76.92%,13.5516
3,2 - Normal Veri (Bayesyen Optuna HPO),LightGBMLSS (Newton GAMLSS),6.6598,23.2392,108.5058,41.35%,9.7350
4,3 - Log Veri ln(y+1) (Varsayılan),XGBoostLSS (Newton GAMLSS),1.9897,19.7140,109.8943,72.12%,9.8532
5,3 - Log Veri ln(y+1) (Varsayılan),LightGBMLSS (Newton GAMLSS),2.1941,19.8585,109.8721,73.08%,10.0188
6,4 - Log Veri ln(y+1) (Bayesyen Optuna HPO),XGBoostLSS (Newton GAMLSS),1.9181,19.6878,109.9681,67.31%,8.5235
7,4 - Log Veri ln(y+1) (Bayesyen Optuna HPO),LightGBMLSS (Newton GAMLSS),1.6541,19.7435,109.9673,85.58%,17.3440


## DeepAR Deneyleri:

### Deney 1:

In [5]:
class DeepARTabular(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super().__init__()
        layers = []
        curr_dim = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(curr_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            curr_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(hidden_dim, 1)
        self.sigma_head = nn.Linear(hidden_dim, 1)
        self.softplus = nn.Softplus()

    def forward(self, x):
        h = self.backbone(x)
        mu = self.mu_head(h)
        sigma = self.softplus(self.sigma_head(h)) + 1e-6
        return mu, sigma

def gaussian_nll_loss(mu, sigma, y):
    var = sigma ** 2
    return torch.mean(0.5 * torch.log(2 * np.pi * var) + 0.5 * ((y - mu) ** 2) / var)

torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_tr = TensorDataset(X_tr_t, y_tr_t)
loader_tr = DataLoader(dataset_tr, batch_size=32, shuffle=True)

model_deepar_def = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=64, num_layers=2)
optimizer = optim.Adam(model_deepar_def.parameters(), lr=0.005)

model_deepar_def.train()
for epoch in range(150):
    for bx, by in loader_tr:
        optimizer.zero_grad()
        mu_b, sig_b = model_deepar_def(bx)
        loss = gaussian_nll_loss(mu_b, sig_b, by)
        loss.backward()
        optimizer.step()

model_deepar_def.eval()
with torch.no_grad():
    mu_te_t, sig_te_t = model_deepar_def(X_te_t)
    
mu_deepar_raw = mu_te_t.squeeze().numpy()
sigma_deepar_raw = sig_te_t.squeeze().numpy()

mc_deepar_raw = stats.norm.rvs(loc=mu_deepar_raw, scale=sigma_deepar_raw, size=(1000, len(mu_deepar_raw)), random_state=42)
y_pred_med_da = np.percentile(mc_deepar_raw, 50, axis=0)
y_pred_low_da = np.percentile(mc_deepar_raw, 5, axis=0)
y_pred_up_da = np.percentile(mc_deepar_raw, 95, axis=0)

mad_da_def = np.median(np.abs(y_test - y_pred_med_da))
mae_da_def = mean_absolute_error(y_test, y_pred_med_da)
rmse_da_def = np.sqrt(mean_squared_error(y_test, y_pred_med_da))
cov_da_def = ((y_test >= y_pred_low_da) & (y_test <= y_pred_up_da)).mean() * 100
width_da_def = np.median(y_pred_up_da - y_pred_low_da)

if 'deepar_results' not in globals():
    deepar_results = []

deepar_results.append({
    "Deney Aşaması": "1 - Normal Veri (Varsayılan)",
    "Model Motoru": "PyTorch DeepAR (Softplus NLL)",
    "MAD (ha)": mad_da_def,
    "MAE (ha)": mae_da_def,
    "RMSE (ha)": rmse_da_def,
    "%90 Kapsama Oranı (%)": cov_da_def,
    "Medyan Bant Genişliği (ha)": width_da_def
})

print(f"DeepAR -> MAD: {mad_da_def:.4f} ha | MAE: {mae_da_def:.4f} ha | RMSE: {rmse_da_def:.4f} ha | %90 Kapsama: %{cov_da_def:.2f} | Genişlik: {width_da_def:.4f} ha")

df_stage1_deepar = pd.DataFrame(deepar_results)
styled_stage1_deepar = (
    df_stage1_deepar.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage1_deepar)

DeepAR -> MAD: 5.2041 ha | MAE: 21.4375 ha | RMSE: 108.4361 ha | %90 Kapsama: %20.19 | Genişlik: 7.3844 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),PyTorch DeepAR (Softplus NLL),5.2041,21.4375,108.4361,20.19%,7.3844


In [22]:
def objective_deepar_raw(trial):
    params = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "lr": trial.suggest_float("lr", 0.001, 0.02, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_t = torch.tensor(y_train[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=params["hidden_dim"], num_layers=params["num_layers"])
        opt = optim.Adam(model.parameters(), lr=params["lr"])
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                m_b, s_b = model(bx)
                l = gaussian_nll_loss(m_b, s_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            m_val, s_val = model(X_val_t)
        mu_val_np = m_val.squeeze().numpy()
        cv_mads.append(np.median(np.abs(y_train[val_idx] - mu_val_np)))
        
    return np.mean(cv_mads)

study_da_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_da_raw.optimize(objective_deepar_raw, n_trials=15, show_progress_bar=False)
best_da_p_raw = study_da_raw.best_params

model_da_opt_raw = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=best_da_p_raw["hidden_dim"], num_layers=best_da_p_raw["num_layers"])
opt_da_raw = optim.Adam(model_da_opt_raw.parameters(), lr=best_da_p_raw["lr"])

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
ds_full_tr = TensorDataset(X_tr_full_t, y_tr_full_t)
ld_full_tr = DataLoader(ds_full_tr, batch_size=32, shuffle=True)

model_da_opt_raw.train()
for epoch in range(120):
    for bx, by in ld_full_tr:
        opt_da_raw.zero_grad()
        m_b, s_b = model_da_opt_raw(bx)
        l = gaussian_nll_loss(m_b, s_b, by)
        l.backward()
        opt_da_raw.step()

model_da_opt_raw.eval()
with torch.no_grad():
    m_te_opt, s_te_opt = model_da_opt_raw(X_te_t)
    
mu_da_opt_np = m_te_opt.squeeze().numpy()
sigma_da_opt_np = s_te_opt.squeeze().numpy()

mc_da_opt_raw = stats.norm.rvs(loc=mu_da_opt_np, scale=sigma_da_opt_np, size=(1000, len(mu_da_opt_np)), random_state=42)
y_med_da_opt = np.percentile(mc_da_opt_raw, 50, axis=0)
y_low_da_opt = np.percentile(mc_da_opt_raw, 5, axis=0)
y_up_da_opt = np.percentile(mc_da_opt_raw, 95, axis=0)

mad_da_opt = np.median(np.abs(y_test - y_med_da_opt))
mae_da_opt = mean_absolute_error(y_test, y_med_da_opt)
rmse_da_opt = np.sqrt(mean_squared_error(y_test, y_med_da_opt))
cov_da_opt = ((y_test >= y_low_da_opt) & (y_test <= y_up_da_opt)).mean() * 100
width_da_opt = np.median(y_up_da_opt - y_low_da_opt)

deepar_results.append({
    "Deney Aşaması": "2 - Normal Veri (Bayesyen Optuna HPO)",
    "Model Motoru": "PyTorch DeepAR (Softplus NLL)",
    "MAD (ha)": mad_da_opt,
    "MAE (ha)": mae_da_opt,
    "RMSE (ha)": rmse_da_opt,
    "%90 Kapsama Oranı (%)": cov_da_opt,
    "Medyan Bant Genişliği (ha)": width_da_opt
})

print(f"DeepAR -> MAD: {mad_da_opt:.4f} ha | MAE: {mae_da_opt:.4f} ha | RMSE: {rmse_da_opt:.4f} ha | %90 Kapsama: %{cov_da_opt:.2f} | Genişlik: {width_da_opt:.4f} ha")

df_stage2_deepar = pd.DataFrame(deepar_results)
styled_stage2_deepar = (
    df_stage2_deepar.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage2_deepar)

DeepAR -> MAD: 3.3732 ha | MAE: 20.2708 ha | RMSE: 109.4123 ha | %90 Kapsama: %59.62 | Genişlik: 8.0484 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),PyTorch DeepAR (Softplus NLL),5.2041,21.4375,108.4361,20.19%,7.3844
1,2 - Normal Veri (Bayesyen Optuna HPO),PyTorch DeepAR (Softplus NLL),3.3732,20.2708,109.4123,59.62%,8.0484


### Deney 3:

In [6]:
torch.manual_seed(42)
np.random.seed(42)

X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)

dataset_tr_log = TensorDataset(X_tr_t, y_tr_log_t)
loader_tr_log = DataLoader(dataset_tr_log, batch_size=32, shuffle=True)

model_da_log_def = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=64, num_layers=2)
opt_da_log = optim.Adam(model_da_log_def.parameters(), lr=0.005)

model_da_log_def.train()
for epoch in range(150):
    for bx, by in loader_tr_log:
        opt_da_log.zero_grad()
        m_b, s_b = model_da_log_def(bx)
        l = gaussian_nll_loss(m_b, s_b, by)
        l.backward()
        opt_da_log.step()

model_da_log_def.eval()
with torch.no_grad():
    m_te_log, s_te_log = model_da_log_def(X_te_t)
    
mu_da_log_np = m_te_log.squeeze().numpy()
sigma_da_log_np = s_te_log.squeeze().numpy()

mc_da_log_raw = stats.norm.rvs(loc=mu_da_log_np, scale=sigma_da_log_np, size=(1000, len(mu_da_log_np)), random_state=42)
mc_da_ha_def = np.clip(np.expm1(mc_da_log_raw), 0, None)

y_med_da_log_def = np.percentile(mc_da_ha_def, 50, axis=0)
y_low_da_log_def = np.percentile(mc_da_ha_def, 5, axis=0)
y_up_da_log_def = np.percentile(mc_da_ha_def, 95, axis=0)

mad_da_log_def = np.median(np.abs(y_test - y_med_da_log_def))
mae_da_log_def = mean_absolute_error(y_test, y_med_da_log_def)
rmse_da_log_def = np.sqrt(mean_squared_error(y_test, y_med_da_log_def))
cov_da_log_def = ((y_test >= y_low_da_log_def) & (y_test <= y_up_da_log_def)).mean() * 100
width_da_log_def = np.median(y_up_da_log_def - y_low_da_log_def)

deepar_results.append({
    "Deney Aşaması": "3 - Log Veri ln(y+1) (Varsayılan)",
    "Model Motoru": "PyTorch DeepAR (Softplus NLL)",
    "MAD (ha)": mad_da_log_def,
    "MAE (ha)": mae_da_log_def,
    "RMSE (ha)": rmse_da_log_def,
    "%90 Kapsama Oranı (%)": cov_da_log_def,
    "Medyan Bant Genişliği (ha)": width_da_log_def
})

print(f"DeepAR -> MAD: {mad_da_log_def:.4f} ha | MAE: {mae_da_log_def:.4f} ha | RMSE: {rmse_da_log_def:.4f} ha | %90 Kapsama: %{cov_da_log_def:.2f} | Genişlik: {width_da_log_def:.4f} ha")

df_stage3_deepar = pd.DataFrame(deepar_results)
styled_stage3_deepar = (
    df_stage3_deepar.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_stage3_deepar)

DeepAR -> MAD: 2.1707 ha | MAE: 19.9158 ha | RMSE: 109.8301 ha | %90 Kapsama: %50.96 | Genişlik: 3.7980 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),PyTorch DeepAR (Softplus NLL),5.2041,21.4375,108.4361,20.19%,7.3844
1,3 - Log Veri ln(y+1) (Varsayılan),PyTorch DeepAR (Softplus NLL),2.1707,19.9158,109.8301,50.96%,3.7980


### Deney 4:

In [7]:
def objective_deepar_log(trial):
    params = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "lr": trial.suggest_float("lr", 0.001, 0.02, log=True)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_t = torch.tensor(X_train_scaled[tr_idx], dtype=torch.float32)
        y_tr_log_t = torch.tensor(y_train_log[tr_idx], dtype=torch.float32).unsqueeze(1)
        X_val_t = torch.tensor(X_train_scaled[val_idx], dtype=torch.float32)
        
        ds_tr = TensorDataset(X_tr_t, y_tr_log_t)
        ld_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
        
        model = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=params["hidden_dim"], num_layers=params["num_layers"])
        opt = optim.Adam(model.parameters(), lr=params["lr"])
        
        model.train()
        for epoch in range(100):
            for bx, by in ld_tr:
                opt.zero_grad()
                m_b, s_b = model(bx)
                l = gaussian_nll_loss(m_b, s_b, by)
                l.backward()
                opt.step()
                
        model.eval()
        with torch.no_grad():
            m_val, s_val = model(X_val_t)
        mu_log_val = m_val.squeeze().numpy()
        preds_ha = np.clip(np.expm1(mu_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha)))
        
    return np.mean(cv_mads)

study_da_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_da_log.optimize(objective_deepar_log, n_trials=15, show_progress_bar=False)
best_da_p_log = study_da_log.best_params

model_da_opt_log = DeepARTabular(input_dim=X_train_scaled.shape[1], hidden_dim=best_da_p_log["hidden_dim"], num_layers=best_da_p_log["num_layers"])
opt_da_log_opt = optim.Adam(model_da_opt_log.parameters(), lr=best_da_p_log["lr"])

X_tr_full_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_full_log_t = torch.tensor(y_train_log, dtype=torch.float32).unsqueeze(1)
ds_full_tr_log = TensorDataset(X_tr_full_t, y_tr_full_log_t)
ld_full_tr_log = DataLoader(ds_full_tr_log, batch_size=32, shuffle=True)

model_da_opt_log.train()
for epoch in range(120):
    for bx, by in ld_full_tr_log:
        opt_da_log_opt.zero_grad()
        m_b, s_b = model_da_opt_log(bx)
        l = gaussian_nll_loss(m_b, s_b, by)
        l.backward()
        opt_da_log_opt.step()

model_da_opt_log.eval()
with torch.no_grad():
    m_te_opt_log, s_te_opt_log = model_da_opt_log(X_te_t)
    
mu_da_opt_log_np = m_te_opt_log.squeeze().numpy()
sigma_da_opt_log_np = s_te_opt_log.squeeze().numpy()

mc_da_opt_log = stats.norm.rvs(loc=mu_da_opt_log_np, scale=sigma_da_opt_log_np, size=(1000, len(mu_da_opt_log_np)), random_state=42)
mc_da_ha_opt = np.clip(np.expm1(mc_da_opt_log), 0, None)

y_med_da_opt_log = np.percentile(mc_da_ha_opt, 50, axis=0)
y_low_da_opt_log = np.percentile(mc_da_ha_opt, 5, axis=0)
y_up_da_opt_log = np.percentile(mc_da_ha_opt, 95, axis=0)

mad_da_opt_log = np.median(np.abs(y_test - y_med_da_opt_log))
mae_da_opt_log = mean_absolute_error(y_test, y_med_da_opt_log)
rmse_da_opt_log = np.sqrt(mean_squared_error(y_test, y_med_da_opt_log))
cov_da_opt_log = ((y_test >= y_low_da_opt_log) & (y_test <= y_up_da_opt_log)).mean() * 100
width_da_opt_log = np.median(y_up_da_opt_log - y_low_da_opt_log)

deepar_results.append({
    "Deney Aşaması": "4 - Log Veri ln(y+1) (Bayesyen Optuna HPO)",
    "Model Motoru": "PyTorch DeepAR (Softplus NLL)",
    "MAD (ha)": mad_da_opt_log,
    "MAE (ha)": mae_da_opt_log,
    "RMSE (ha)": rmse_da_opt_log,
    "%90 Kapsama Oranı (%)": cov_da_opt_log,
    "Medyan Bant Genişliği (ha)": width_da_opt_log
})

print(f"DeepAR -> MAD: {mad_da_opt_log:.4f} ha | MAE: {mae_da_opt_log_log if 'mae_da_opt_log_log' in locals() else mae_da_opt_log:.4f} ha | RMSE: {rmse_da_opt_log:.4f} ha | %90 Kapsama: %{cov_da_opt_log:.2f} | Genişlik: {width_da_opt_log:.4f} ha")

df_deepar_final = pd.DataFrame(deepar_results)
styled_deepar_final = (
    df_deepar_final.style
    .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Oranges")
    .background_gradient(subset=["%90 Kapsama Oranı (%)"], cmap="Blues")
    .format({
        "MAD (ha)": "{:.4f}",
        "MAE (ha)": "{:.4f}",
        "RMSE (ha)": "{:.4f}",
        "%90 Kapsama Oranı (%)": "{:.2f}%",
        "Medyan Bant Genişliği (ha)": "{:.4f}"
    })
    .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
    .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#2c3e50'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
    ])
)
display(styled_deepar_final)

DeepAR -> MAD: 2.7362 ha | MAE: 20.5761 ha | RMSE: 110.2326 ha | %90 Kapsama: %50.96 | Genişlik: 4.4961 ha


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),%90 Kapsama Oranı (%),Medyan Bant Genişliği (ha)
0,1 - Normal Veri (Varsayılan),PyTorch DeepAR (Softplus NLL),5.2041,21.4375,108.4361,20.19%,7.3844
1,3 - Log Veri ln(y+1) (Varsayılan),PyTorch DeepAR (Softplus NLL),2.1707,19.9158,109.8301,50.96%,3.7980
2,4 - Log Veri ln(y+1) (Bayesyen Optuna HPO),PyTorch DeepAR (Softplus NLL),2.7362,20.5761,110.2326,50.96%,4.4961
